<a href="https://colab.research.google.com/github/toecm/iedi-mas/blob/main/MA_IEDI_102926.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip MA-IEDI.zip

Archive:  MA-IEDI.zip
   creating: content/MA-IEDI/
  inflating: content/MA-IEDI/config.py  
   creating: content/MA-IEDI/srv/
  inflating: content/MA-IEDI/srv/utils.py  
  inflating: content/MA-IEDI/srv/brain_agent.py  
 extracting: content/MA-IEDI/srv/__init__.py  
  inflating: content/MA-IEDI/srv/input_agent.py  
  inflating: content/MA-IEDI/srv/ux_agent.py  
 extracting: content/MA-IEDI/srv/managers.py  
   creating: content/MA-IEDI/srv/.ipynb_checkpoints/
  inflating: content/MA-IEDI/srv/trust_agent.py  
  inflating: content/MA-IEDI/main.py  
   creating: content/MA-IEDI/.ipynb_checkpoints/
   creating: content/MA-IEDI/src/
  inflating: content/MA-IEDI/src/utils.py  
  inflating: content/MA-IEDI/src/brain_agent.py  
 extracting: content/MA-IEDI/src/__init__.py  
  inflating: content/MA-IEDI/src/input_agent.py  
  inflating: content/MA-IEDI/src/ux_agent.py  
  inflating: content/MA-IEDI/src/managers.py  
   creating: content/MA-IEDI/src/.ipynb_checkpoints/
  inflating: content/MA-I

In [ ]:
!pip install -q openai-whisper rapidfuzz pandas gradio datasets transformers torchaudio torch librosa pydub ffmpeg-python jiwer google-genai python-dotenv requests yt-dlp soundfile web3 eth-account huggingface_hub
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
%cd MA-IEDI
!python main.py

[Errno 2] No such file or directory: 'MA-IEDI'
/content
python3: can't open file '/content/main.py': [Errno 2] No such file or directory


In [ ]:
import shutil
import os

# 1. Force delete the confusing directory
if os.path.exists("/content/MA-IEDI"):
    shutil.rmtree("/content/MA-IEDI")
    print("🗑️ Deleted confused directory.")

# 2. Re-unzip cleanly
# Make sure your zip file is actually named MA-IEDI.zip
!unzip -q /content/MA-IEDI.zip -d /content/

print("✅ Folder restored. You should now see /content/MA-IEDI/")

replace /content/content/MA-IEDI/config.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/content/MA-IEDI/srv/utils.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# DIAGNOSTIC: List my available models
from google import genai
client = genai.Client(api_key=GOOGLE_API_KEY)
print("🔍 Scanning available models...")
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(f" - {m.name}")

NameError: name 'GOOGLE_API_KEY' is not defined

## 🚀 Getting Started with Hardhat

Hardhat is a development environment for compiling, deploying, testing, and debugging your Ethereum software. It helps developers manage and automate the recurring tasks that are inherent to building smart contracts and dApps.

### 1. Install Node.js and npm (if you don't have them)
Hardhat projects are typically set up using Node.js and its package manager, `npm`. You can download Node.js (which includes npm) from the official website: [nodejs.org](https://nodejs.org/en/download/).

### 2. Create a New Project Directory
It's best to create a dedicated directory for your Hardhat project.


In [ ]:
import os

project_name = "my-hardhat-project"
if not os.path.exists(project_name):
    os.makedirs(project_name)
    print(f"Created directory: {project_name}")
else:
    print(f"Directory '{project_name}' already exists.")

# Change to the new directory
%cd {project_name}

### 3. Initialize the Project and Install Hardhat

Inside your project directory, you'll initialize a new npm project and then install Hardhat locally.


In [ ]:
!npm init -y
!npm install --save-dev hardhat

### 4. Create a Hardhat Project

Now you can run the Hardhat command to create your first project. It will ask you to choose a project type (e.g., "Create a basic sample project"). You can select the default options.


In [ ]:
!npx hardhat

After running `npx hardhat`, you'll have a basic project structure with sample contracts, scripts, and tests. You can explore these files in the file browser (`/content/my-hardhat-project`).

### Next Steps:
*   **Explore `hardhat.config.js`**: This is where you configure your network, compilers, and plugins.
*   **Write Smart Contracts**: Look into the `contracts/` directory to start writing your Solidity code.
*   **Write Tests**: Use the `test/` directory to write tests for your contracts.
*   **Run Scripts**: The `scripts/` directory is for deployment and interaction scripts.

Let me know if you want to compile, deploy, or interact with a sample contract!

In [ ]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv

# Load keys
load_dotenv()
PINATA_JWT = os.getenv("PINATA_JWT")

def fetch_ipfs_logs():
    if not PINATA_JWT:
        print("❌ Error: PINATA_JWT not found.")
        return

    print("🔍 Fetching pinned files from Pinata...")

    url = "https://api.pinata.cloud/data/pinList?status=pinned"
    headers = {"Authorization": f"Bearer {PINATA_JWT}"}

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        files = response.json().get('rows', [])

        print(f"✅ Found {len(files)} pinned logs.")

        all_logs = []

        for file in files:
            cid = file['ipfs_pin_hash']
            # Fetch content from a public gateway
            gateway_url = f"https://gateway.pinata.cloud/ipfs/{cid}"
            try:
                log_data = requests.get(gateway_url).json()
                # Add CID for reference
                log_data['ipfs_cid'] = cid
                all_logs.append(log_data)
                print(f"   -> Retrieved log: {cid}")
            except Exception as e:
                print(f"   ⚠️ Could not read content for {cid}: {e}")

        # Convert to DataFrame for easy viewing
        if all_logs:
            df = pd.DataFrame(all_logs)
            print("\n📊 Retrieved Data Summary:")
            print(df.head())

            # Save to CSV for analysis
            df.to_csv("ipfs_audit_trail.csv", index=False)
            print("\n💾 Saved full log to 'ipfs_audit_trail.csv'")
            return df
        else:
            print("No valid logs found.")

    except Exception as e:
        print(f"❌ API Error: {e}")

# Run the retrieval
audit_df = fetch_ipfs_logs()